In [2]:
!git clone https://github.com/AI-FREE-Team/Traditional-Chinese-Handwriting-Dataset.git
import os, zipfile, shutil

os.chdir('/content/Traditional-Chinese-Handwriting-Dataset/data')
for item in os.listdir():
    if item.endswith('.zip'):
        print(f"Unzipping {item}...")
        with zipfile.ZipFile(item, 'r') as zip_ref:
            zip_ref.extractall('/content')
print("Done!")

fatal: destination path 'Traditional-Chinese-Handwriting-Dataset' already exists and is not an empty directory.
Unzipping cleaned_data(50_50)-20200420T071507Z-004.zip...
Unzipping cleaned_data(50_50)-20200420T071507Z-001.zip...
Unzipping cleaned_data(50_50)-20200420T071507Z-002.zip...
Unzipping cleaned_data(50_50)-20200420T071507Z-003.zip...
Done!


In [3]:
data_path = '/content/cleaned_data(50_50)'
print("Number of characters:", len(os.listdir(data_path)))
print("Example folders:", sorted(os.listdir(data_path))[:10])

Number of characters: 250712
Example folders: ['丁_0.png', '丁_1.png', '丁_10.png', '丁_11.png', '丁_12.png', '丁_13.png', '丁_14.png', '丁_15.png', '丁_16.png', '丁_17.png']


In [4]:
DATA_DIR = '/content/cleaned_data(50_50)'
IMG_SIZE = (64, 64)
MAX_CLASSES = 200
BATCH_SIZE = 64
EPOCHS = 15

In [5]:
import os
import shutil
from collections import defaultdict

src = '/content/cleaned_data(50_50)'
dst = '/content/organized_data'

# Create new clean folder
if os.path.exists(dst):
    shutil.rmtree(dst)
os.makedirs(dst)

# Group images by character
char_images = defaultdict(list)

for filename in os.listdir(src):
    if filename.endswith(('.png', '.jpg', '.jpeg')):
        # Character is the part before the last underscore
        char = filename.rsplit('_', 1)[0]
        char_images[char].append(filename)

print(f"Found {len(char_images)} characters")

# Move images into character folders
for char, files in char_images.items():
    char_dir = os.path.join(dst, char)
    os.makedirs(char_dir, exist_ok=True)
    for f in files:
        shutil.move(os.path.join(src, f), os.path.join(char_dir, f))

print("Reorganization done!")
print("Number of character folders:", len(os.listdir(dst)))
print("Example folders:", sorted(os.listdir(dst))[:10])

Found 4803 characters
Reorganization done!
Number of character folders: 4803
Example folders: ['丁', '七', '丈', '三', '上', '下', '不', '丐', '丑', '且']


In [6]:
data_path = '/content/organized_data'
print("Number of characters:", len(os.listdir(data_path)))
print("Example folders:", sorted(os.listdir(data_path))[:15])

# Check one character
example_char = sorted(os.listdir(data_path))[0]
print(f"\nImages inside '{example_char}':", len(os.listdir(os.path.join(data_path, example_char))))
print(os.listdir(os.path.join(data_path, example_char))[:5])

Number of characters: 4803
Example folders: ['丁', '七', '丈', '三', '上', '下', '不', '丐', '丑', '且', '丕', '世', '丘', '丙', '丞']

Images inside '丁': 51
['丁_2.png', '丁_11.png', '丁_36.png', '丁_44.png', '丁_30.png']


In [7]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from pathlib import Path
import random

# ====================== Config ======================
DATA_DIR = '/content/organized_data'
IMG_SIZE = (64, 64)
MAX_CLASSES = 200          # Start with 200 classes (change to None later for all 4803)
NUM_TRAIN_PER_CLASS = 40   # First 40 images for training
BATCH_SIZE = 64
EPOCHS = 15
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ====================== Load data ======================
def load_data(data_dir, img_size, max_classes=None, num_train=40):
    classes = sorted([d for d in os.listdir(data_dir)
                      if os.path.isdir(os.path.join(data_dir, d))])

    if max_classes is not None:
        classes = classes[:max_classes]

    print(f"Using {len(classes)} classes")

    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

    X_train, y_train = [], []
    X_test, y_test = [], []

    for cls in classes:
        cls_path = Path(data_dir) / cls
        images = sorted(list(cls_path.glob("*.png")) + list(cls_path.glob("*.jpg")))

        # First 40 → train, rest → test
        train_imgs = images[:num_train]
        test_imgs = images[num_train:]

        for img_path in train_imgs:
            img = load_img(img_path, color_mode="grayscale", target_size=img_size)
            X_train.append(img_to_array(img) / 255.0)
            y_train.append(class_to_idx[cls])

        for img_path in test_imgs:
            img = load_img(img_path, color_mode="grayscale", target_size=img_size)
            X_test.append(img_to_array(img) / 255.0)
            y_test.append(class_to_idx[cls])

    return (np.array(X_train), np.array(y_train),
            np.array(X_test), np.array(y_test),
            classes)

print("Loading data... (this may take 1-2 minutes)")
X_train, y_train, X_test, y_test, classes = load_data(
    DATA_DIR, IMG_SIZE, max_classes=MAX_CLASSES, num_train=NUM_TRAIN_PER_CLASS
)

print(f"\nTrain samples: {len(X_train)}")
print(f"Test samples : {len(X_test)}")
print(f"Number of classes: {len(classes)}")
print(f"Image shape: {X_train[0].shape}")

Loading data... (this may take 1-2 minutes)
Using 200 classes

Train samples: 7996
Test samples : 2187
Number of classes: 200
Image shape: (64, 64, 1)


In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("Using device:", tf.config.list_physical_devices('GPU'))

# ========== Faster Config ==========
EPOCHS = 10
BATCH_SIZE = 128          # bigger batch = faster
STEPS = 80                # much fewer steps per epoch

# ========== Augmentation ==========
train_datagen = ImageDataGenerator(
    rotation_range=12,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    fill_mode='nearest'
)

train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)

# ========== Fast Models ==========
def build_fast_cnn():
    model = models.Sequential([
        layers.Input(shape=(64, 64, 1)),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(len(classes), activation='softmax')
    ], name='fast_cnn')
    return model

def build_faster_cnn():
    model = models.Sequential([
        layers.Input(shape=(64, 64, 1)),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(len(classes), activation='softmax')
    ], name='simple_fast')
    return model

def train_fast(model, name):
    print(f"\n===== Training {name} =====")
    model.compile(
        optimizer=optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        train_gen,
        steps_per_epoch=STEPS,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[
            callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2)
        ],
        verbose=1
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f">>> {name} Test Accuracy: {acc*100:.2f}%")
    model.save(f'{name}.keras')
    return acc

# ========== Train only 2 models ==========
results = {}

model1 = build_fast_cnn()
results['fast_cnn'] = train_fast(model1, 'fast_cnn')

model2 = build_faster_cnn()
results['simple_fast'] = train_fast(model2, 'simple_fast')

print("\n" + "="*50)
print("FINAL RESULTS (200 classes)")
print("="*50)
for k, v in results.items():
    print(f"{k}: {v*100:.2f}%")

Using device: []

===== Training fast_cnn =====
Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 83s 1s/step - accuracy: 0.0043 - loss: 5.3002 - val_accuracy: 0.0050 - val_loss: 5.2983 - learning_rate: 0.0010
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.0051 - loss: 5.2991 - val_accuracy: 0.0050 - val_loss: 5.2969 - learning_rate: 0.0010
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.0036 - loss: 5.2874 - val_accuracy: 0.0050 - val_loss: 5.2132 - learning_rate: 0.0010
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.0093 - loss: 5.1219 - val_accuracy: 0.0110 - val_loss: 6.0337 - learning_rate: 5.0000e-04
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.0119 - loss: 4.9099 - val_accuracy: 0.0078 - val_loss: 5.9147 - learning_rate: 5.0000e-04
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.0136 - loss: 4.8206 - val_accuracy: 0.0101 - val_loss: 6.1605 - learning_rate: 5.0000e-04
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 85

In [10]:
def build_better_cnn():
    model = models.Sequential([
        layers.Input(shape=(64, 64, 1)),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(len(classes), activation='softmax')
    ], name='better_cnn')
    return model

model3 = build_better_cnn()
results['better_cnn'] = train_fast(model3, 'better_cnn')
print("better_cnn:", results['better_cnn']*100)


===== Training better_cnn =====
Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 130s 2s/step - accuracy: 0.0056 - loss: 5.4113 - val_accuracy: 0.0050 - val_loss: 9.4598 - learning_rate: 0.0010
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.0054 - loss: 5.2916 - val_accuracy: 0.0050 - val_loss: 11.6920 - learning_rate: 0.0010
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.0088 - loss: 5.2476 - val_accuracy: 0.0055 - val_loss: 10.8509 - learning_rate: 0.0010
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 139s 2s/step - accuracy: 0.0104 - loss: 5.1601 - val_accuracy: 0.0055 - val_loss: 8.5600 - learning_rate: 0.0010
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.0114 - loss: 5.0894 - val_accuracy: 0.0096 - val_loss: 8.1770 - learning_rate: 0.0010
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.0113 - loss: 5.0496 - val_accuracy: 0.0101 - val_loss: 5.3086 - learning_rate: 0.0010
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accura